### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GITHUB_API_KEY"]=os.getenv("GITHUB_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [3]:
# Step1 : Load and chunk the document
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

In [4]:
# Step 2 : Create embeddings and store in FAISS
embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [7]:
# Step 3 :Create MMR retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

In [8]:
# Step 4: Prompt and LLM
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")

from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = init_chat_model(
    model="groq/compound-mini",
    model_provider="groq",
    api_key=os.getenv("GROQ_API_KEY"),
)
llm

ChatGroq(profile={}, client=<groq.resources.chat.completions.Completions object at 0x000001A337ABF7A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A360E9FBF0>, model_name='groq/compound-mini', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
# Step 5 : Create RAG pipeline
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain = create_retrieval_chain(retriever=mmr_retriever,combine_docs_chain=document_chain)

In [11]:
# Step 6: Query
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("Answer:\n", response["answer"])

Answer:
 LangChain builds two complementary capabilities into its framework:

### Agents  
* **Tool‑driven reasoning** – An agent can be given a set of tools (e.g., a calculator, a web‑search API, custom functions, database connectors, etc.).  
* **Dynamic tool selection** – The LLM acting as the agent decides, step‑by‑step, which tool to invoke and in what order, based on the instructions it receives and the intermediate results it observes.  
* **External integration** – Because the tools can wrap external APIs or databases, agents can fetch live data, run computations, or perform any custom operation, extending the raw language model far beyond text generation.

### Memory  
* **Conversation continuity** – LangChain provides built‑in memory objects that store what has been said so far, allowing the model to reference prior turns and produce coherent multi‑turn dialogs.  
* **Memory types** –  
  * **`ConversationBufferMemory`** keeps a raw transcript of the dialogue (or a configurab

In [12]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='3890c2af-4038-4f50-aa5a-1d8d275844d6', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='d5aeda5d-2219-4d88-a3d4-ead6659edc6d', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered applications.\nRAG pipelines in LangChain involve document loading, splitting, embedding, retrieval, and LLM-based response generation.'),
  Document(id='8ee2cda0-89a1-40cc-884b-eb7154a2b7bb', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain allows LLMs to act as agents that decide which tool to call and in what order duri